## MAKING AND WRITING MEMORY

In [144]:
# !pip install ollama numpy   # <- uncomment and run once if needed
import ollama, json
import numpy as np
from datetime import datetime, timezone

CHAT_MODEL  = "gemma4:31b-cloud"        
EMBED_MODEL = "embeddinggemma:latest"

In [145]:
def embed(text):
    """Turn text into a meaning-vector using nomic-embed-text."""
    return np.array(ollama.embeddings(model=EMBED_MODEL, prompt=text)["embedding"], dtype=float)

print("helpers ready: chat(), embed()")

helpers ready: chat(), embed()


### CREATING AN CONVERSATION


In [146]:
import ollama

messages = []   # universal conversation store

def chat_session():
    print("(type 'bye' to end)\n")

    while True:
        user_text = input("You: ")

        if user_text.lower() == "bye":
            break

        # Add user message to conversation history
        messages.append({
            "role": "user",
            "content": user_text
        })

        # Directly call Ollama
        response = ollama.chat(
            model=CHAT_MODEL,
            messages=messages
        )
        # Extract assistant's response
        reply = response["message"]["content"]

        # Add assistant response to conversation history
        messages.append({
            "role": "assistant",
            "content": reply
        })

        print("Assistant:", reply, "\n")


print("chat_session() ready -> type 'bye' to stop")
chat_session()

chat_session() ready -> type 'bye' to stop
(type 'bye' to end)



You:  Hi, I'm Ajay, a data science instructor based in Delhi.


Assistant: Hello Ajay! It's a pleasure to meet you. 

It sounds like you're doing some exciting work in Delhi. Given the booming tech scene and the huge number of students and professionals in the NCR region looking to upskill, being a data science instructor must keep you very busy!

Whether you're looking to brainstorm new curriculum ideas, need help explaining a complex machine learning concept to your students, or want to automate some of your grading or coding examples, I'm here to help.

What are you focusing on teaching at the moment? (e.g., Python basics, Deep Learning, GenAI, or maybe preparing students for interviews?) 



You:  Delhi is capital of which country.


Assistant: Delhi is the capital of **India**. 



You:  I'm allergic to peanuts, so keep that in mind for any food suggestions.


... (10 outputs left out)

Assistant: Noted, Ajay. I've added that to your preferences: **Python is your primary language, and you prefer it over R.**

Whenever you need code for your Context Engineering course or any other data science project, I will provide it in Python.

Since you're building that course for CampusX, you're likely using libraries like **LangChain, LlamaIndex, or Haystack** to handle context management. If you need any help building modular, step-by-step Python examples for those, just let me know! 



You:  What is difference between python and R?


Assistant: Since you already prefer Python, you likely know its strengths, but when explaining this to your students at CampusX, it's helpful to frame it as a difference in **purpose** and **philosophy**.

Here is the breakdown of the key differences:

### 1. Primary Purpose
*   **Python:** A **General-Purpose Language**. It was designed to be a versatile tool. You can use it for web development (Django/Flask), automation, software engineering, and—of course—Data Science and AI.
*   **R:** A **Statistical Language**. It was built by statisticians, for statisticians. Its primary goal is data analysis, complex statistical modeling, and high-quality data visualization.

### 2. Learning Curve & Syntax
*   **Python:** Known for its "readability." The syntax is very close to English, which makes it easier for beginners to pick up. It follows a strict structure (like indentation) that encourages clean code.
*   **R:** Has a steeper learning curve for those who aren't from a math/stats backgro

You:  BYE


In [ ]:
demo_inputs = [
    "Hi, I'm Ajay, a data science instructor based in Delhi.",
    "Delhi is capital of which country.",
    "I'm allergic to peanuts, so keep that in mind for any food suggestions.",
    "Do peanuts contains protein.",
    "Whenever you show me code, explain it step by step — I hate big code dumps.",
    "Last week I started building a Context Engineering course for CampusX.",
    "By the way, what's the capital of France?",
    "I mostly work in Python and prefer it over R.",
    "What is difference between python and R?"
]

In [147]:
messages

[{'role': 'user',
  'content': "Hi, I'm Ajay, a data science instructor based in Delhi."},
 {'role': 'assistant',
  'content': "Hello Ajay! It's a pleasure to meet you. \n\nIt sounds like you're doing some exciting work in Delhi. Given the booming tech scene and the huge number of students and professionals in the NCR region looking to upskill, being a data science instructor must keep you very busy!\n\nWhether you're looking to brainstorm new curriculum ideas, need help explaining a complex machine learning concept to your students, or want to automate some of your grading or coding examples, I'm here to help.\n\nWhat are you focusing on teaching at the moment? (e.g., Python basics, Deep Learning, GenAI, or maybe preparing students for interviews?)"},
 {'role': 'user', 'content': 'Delhi is capital of which country.'},
 {'role': 'assistant', 'content': 'Delhi is the capital of **India**.'},
 {'role': 'user',
  'content': "I'm allergic to peanuts, so keep that in mind for any food sugge

## CREATE STORAGE

In [130]:
class SemanticMemory:
    """Stable FACTS about the user (who they are / what's true).
    Built as a dict. Written by upsert (overwrite by label)."""
    def __init__(self):
        self._facts = {}                 # empty dict is created when the object is made

    def upsert(self, key, value):        # upsert = UPDATE if key exists, INSERT if new
        self._facts[key] = value

    def get(self, key):                  # read ONE fact by its label (None if missing)
        return self._facts.get(key)

    def delete(self, key):               # remove a fact (needed when a fact stops being true)
        self._facts.pop(key, None)

    def get_all(self):
        return dict(self._facts)         # return a COPY of the whole box (no search)


class EpisodicMemory:
    """EVENTS that happened (usually time-stamped).
    Built as a list; each item stores an embedding so it's searchable later."""
    def __init__(self):
        self._events = []                # empty list is created when the object is made

    def add(self, text):
        self._events.append({            # APPEND -- old events are never rewritten
            "text": text,
            "time": datetime.now(timezone.utc).isoformat(),
            "vector": embed(text),
        })

    def get_all(self):                   # matching interface: hand back all events
        return list(self._events)        # return a COPY of the pile


class ProceduralMemory:
    """Learned RULES about how to help this user (how to behave).
    Its OWN store. Behaves like SemanticMemory (small, stable, always-on),
    but kept separate so the three types stay clearly distinct."""
    def __init__(self):
        self._rules = {}

    def upsert(self, key, value):        # same overwrite behaviour as facts
        self._rules[key] = value

    def get(self, key):                  # read ONE rule by its label
        return self._rules.get(key)

    def delete(self, key):               # remove a rule (when a preference changes)
        self._rules.pop(key, None)

    def get_all(self):
        return dict(self._rules)


print("Stores ready: SemanticMemory (facts) | EpisodicMemory (events) | ProceduralMemory (rules)")

Stores ready: SemanticMemory (facts) | EpisodicMemory (events) | ProceduralMemory (rules)


## EXTRACTING MEMORIES THAT WILL BE STORED.

In [148]:
# ---------- Step 1: EXTRACTION (the write decision) ----------

# The instruction that teaches the model WHAT to pull out and HOW to classify it.
# This is static -> it goes in the system role. The changing conversation goes in
# the user role (keeps the system prompt a stable, cacheable prefix).
EXTRACTION_PROMPT = """You extract long-term MEMORY from a conversation — durable facts,
events, and preferences that will help in FUTURE, separate conversations with this user.

Save something only if it will still matter days from now. Ignore greetings, small talk,
one-off questions the user just wanted answered, and anything temporary.

Classify each memory as exactly one type:

- "semantic"  = a lasting FACT about the user (who they are / what's true).
                Stays true tomorrow, no date attached.
                e.g. "Works as a data scientist", "Allergic to peanuts", "Prefers Python"

- "episodic"  = a specific EVENT that happened, tied to a moment in time.
                Describes something that occurred, not something that just is.
                e.g. "Fixed a login bug last Tuesday", "Started a new course last week"

- "procedural"= a RULE about HOW to help this user (their preferred way of being helped).
                An instruction to follow, not a fact about them.
                e.g. "Explain code step by step", "Keep answers short"

Write each memory from the user's perspective: turn "I" into "the user", resolve vague
references, and make each item a single self-contained sentence.

Return ONLY a JSON list. Each item:
  "type":    "semantic" | "episodic" | "procedural"
  "key":     a short snake_case label for semantic/procedural; use "" for episodic
  "content": one short sentence describing the memory

Example
Conversation:
User: Hey! I'm a data scientist, and last week I finally shipped my first ML model to production.
Also please explain code to me step by step, not all at once.
Output:
[
  {"type":"semantic","key":"profession","content":"Works as a data scientist"},
  {"type":"episodic","key":"","content":"Shipped their first ML model to production last week"},
  {"type":"procedural","key":"code_style","content":"Explain code step by step, not all at once"}
]

If nothing is worth remembering, return []."""


def _extract_json(text):
    """Models often wrap JSON in ```json fences or add stray words. Clean it up
    so json.loads() can parse it."""
    t = text.strip()
    if t.startswith("```"):                       # strip a ```json ... ``` fence
        t = t.split("```")[1]
        if t.lower().startswith("json"):
            t = t[4:]
        t = t.strip()
    if not t.startswith("["):                      # fallback: grab first [ ... last ]
        i, j = t.find("["), t.rfind("]")
        if i != -1 and j != -1:
            t = t[i:j + 1]
    return t


def extract_memories(conversation, system_prompt=EXTRACTION_PROMPT):
    """Ask the model to read a conversation and return memory candidates.
    Returns (items, tokens):
      items  -> list of {"type","key","content"} dicts
      tokens -> real prompt token count from Ollama (we care about numbers here)."""
    response = ollama.chat(
        model=CHAT_MODEL,
        messages=[
            {"role": "system", "content": system_prompt},                        # static instruction
            {"role": "user",   "content": f"Conversation:\n{conversation}\nOutput:"}  # changing data
        ],
        format="json",   # ask Ollama to guarantee valid JSON -> far fewer parse failures
    )

    reply  = response["message"]["content"]        # the model's text answer
    try:
        items = json.loads(_extract_json(reply))
    except Exception:
        print("Could not parse JSON. Raw reply was:\n", reply)
        items = []

    return items


print("extract_memories() ready")

extract_memories() ready


In [149]:
extract_memories(messages)

[{'type': 'semantic',
  'key': 'user_identity',
  'content': 'The user is Ajay, a data science instructor based in Delhi.'},
 {'type': 'semantic',
  'key': 'dietary_restriction',
  'content': 'The user is allergic to peanuts.'},
 {'type': 'procedural',
  'key': 'code_explanation_style',
  'content': 'Explain code step by step and avoid providing large code dumps.'},
 {'type': 'episodic',
  'key': '',
  'content': 'The user started building a Context Engineering course for CampusX last week.'},
 {'type': 'semantic',
  'key': 'programming_preference',
  'content': 'The user mostly works in Python and prefers it over R.'}]

In [151]:
candidates= extract_memories(messages)

In [152]:
candidates

[{'type': 'semantic',
  'key': 'user_name',
  'content': "The user's name is Ajay."},
 {'type': 'semantic',
  'key': 'profession',
  'content': 'Works as a data science instructor based in Delhi.'},
 {'type': 'semantic',
  'key': 'health_allergy',
  'content': 'The user is allergic to peanuts.'},
 {'type': 'semantic',
  'key': 'language_preference',
  'content': 'Works mostly in Python and prefers it over R.'},
 {'type': 'procedural',
  'key': 'code_explanation_style',
  'content': 'Explain code step by step and avoid large code dumps.'},
 {'type': 'episodic',
  'key': '',
  'content': 'Started building a Context Engineering course for CampusX last week.'}]

## STORING INFORMATION IN RESPECTIVE STORAGE

In [153]:
#FUNCTION TO ROUTE THE MESSAGES TO THE RESPECTIVE STORES
def store_memory(item, semantic, episodic, procedural):
    """Route ONE candidate to the right store, in that store's write style."""
    t = item.get("type")
    if t == "semantic":     
        semantic.upsert(item["key"], item["content"])    # fact  -> semantic store [USING UPSERT FUNCTION]
    elif t == "episodic":   
        episodic.add(item["content"])                     # event -> episodic store
    elif t == "procedural": 
        procedural.upsert(item["key"], item["content"])   # rule  -> procedural store
    else:                   
        print("Unknown type, skipped:", item)


In [154]:
semantic   = SemanticMemory()
episodic   = EpisodicMemory()
procedural = ProceduralMemory()


for item in candidates:
     store_memory(item, semantic, episodic, procedural)
    

In [155]:
def show_memories(semantic, episodic, procedural):
    """Print the THREE kinds of memory, each from its OWN store."""
    print("SEMANTIC   (facts):")
    for k, v in semantic.get_all().items():   print(f"   {k:<12} -> {v}")
    print("EPISODIC   (events):")
    for e in episodic._events:                print(f"   {e['time'][:10]}   {e['text']}")
    print("PROCEDURAL (learned rules):")
    for k, v in procedural.get_all().items(): print(f"   {k:<12} -> {v}")

In [156]:
show_memories(semantic, episodic, procedural)

SEMANTIC   (facts):
   user_name    -> The user's name is Ajay.
   profession   -> Works as a data science instructor based in Delhi.
   health_allergy -> The user is allergic to peanuts.
   language_preference -> Works mostly in Python and prefers it over R.
EPISODIC   (events):
   2026-08-21   Started building a Context Engineering course for CampusX last week.
PROCEDURAL (learned rules):
   code_explanation_style -> Explain code step by step and avoid large code dumps.


In [157]:
messages_2 = [
    {
        "role": "user",
        "content": "Morning! Quick update from my side — alongside the data science work, I've started freelance consulting for a couple of startups on the side."
    },
    {
        "role": "assistant",
        "content": "That's great — consulting is a nice way to see different problems."
    },
    {
        "role": "user",
        "content": "Yeah, it's been fun. Oh, last Friday I gave a guest lecture at a local college on machine learning basics — the students really enjoyed it."
    },
    {
        "role": "assistant",
        "content": "Nice, guest lectures are always rewarding!"
    },
    {
        "role": "user",
        "content": "One more thing about our sessions — whenever you share a diagram or table, could you always add a one-line caption under it? Helps me a lot."
    },
    {
        "role": "assistant",
        "content": "Got it, I'll add a one-line caption under every diagram or table."
    },
    {
        "role": "user",
        "content": "Perfect, thanks!"
    },
]

In [158]:
print("BEFORE this chat:")
show_memories(semantic, episodic, procedural)

BEFORE this chat:
SEMANTIC   (facts):
   user_name    -> The user's name is Ajay.
   profession   -> Works as a data science instructor based in Delhi.
   health_allergy -> The user is allergic to peanuts.
   language_preference -> Works mostly in Python and prefers it over R.
EPISODIC   (events):
   2026-08-21   Started building a Context Engineering course for CampusX last week.
PROCEDURAL (learned rules):
   code_explanation_style -> Explain code step by step and avoid large code dumps.


In [159]:
# Same pipeline as before. We're not touching the stores by hand -- the chat drives it.
saved_2 = extract_memories(messages_2)

In [160]:
saved_2

[{'type': 'semantic',
  'key': 'side_business',
  'content': 'The user does freelance consulting for startups alongside their data science work.'},
 {'type': 'episodic',
  'key': '',
  'content': 'The user gave a guest lecture on machine learning basics at a local college last Friday.'},
 {'type': 'procedural',
  'key': 'formatting_preference',
  'content': 'Always add a one-line caption under every diagram or table shared.'}]

In [161]:
for item in saved_2:
     store_memory(item, semantic, episodic, procedural)

In [162]:
print(f"\nThe conversation produced {len(saved_2)} memories.\n")
show_memories(semantic, episodic, procedural)


The conversation produced 3 memories.

SEMANTIC   (facts):
   user_name    -> The user's name is Ajay.
   profession   -> Works as a data science instructor based in Delhi.
   health_allergy -> The user is allergic to peanuts.
   language_preference -> Works mostly in Python and prefers it over R.
   side_business -> The user does freelance consulting for startups alongside their data science work.
EPISODIC   (events):
   2026-08-21   Started building a Context Engineering course for CampusX last week.
   2026-08-21   The user gave a guest lecture on machine learning basics at a local college last Friday.
PROCEDURAL (learned rules):
   code_explanation_style -> Explain code step by step and avoid large code dumps.
   formatting_preference -> Always add a one-line caption under every diagram or table shared.
